# 🎬 Rufus — AI YouTube Video Generator

Generate complete YouTube videos from just a **topic** — 100% free on Google Colab!

## 🛠️ Pipeline
| Step | Tool | Cost |
|------|------|------|
| 📝 Script | Google Gemini Flash | Free |
| 🔊 Voiceover | Microsoft Edge TTS | Free |
| 🎨 Images | Stable Diffusion v1.5 | Free (Colab GPU) |
| 🎞️ Assembly | MoviePy + FFmpeg | Free |

## ✅ Setup
1. Get a **free Gemini API key**: [aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey)
2. Enable GPU: `Runtime → Change runtime type → T4 GPU`
3. Run all: `Runtime → Run all`

In [ ]:
# @title 📦 Step 0: Install Dependencies
import subprocess, sys

print("Installing packages (~2 min on first run)...")
r1 = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "google-generativeai", "edge-tts", "moviepy",
     "diffusers", "transformers", "accelerate", "Pillow", "nest_asyncio"],
    capture_output=True, text=True
)
print("✅ Python packages installed" if r1.returncode == 0 else r1.stderr[-300:])

r2 = subprocess.run(["apt-get", "install", "-y", "-q", "ffmpeg"], capture_output=True, text=True)
print("✅ FFmpeg ready" if r2.returncode == 0 else "ℹ️ FFmpeg already present")
print("\n🚀 Ready!")

In [ ]:
# @title ⚙️ Configuration { display-mode: "form" }
# @markdown ### 🔑 Gemini API Key (free at aistudio.google.com/app/apikey)
GEMINI_API_KEY = ""  # @param {type:"string"}
# @markdown ---
# @markdown ### 🎬 Video Settings
VIDEO_TOPIC = "5 amazing productivity hacks you need to know"  # @param {type:"string"}
NUM_SCENES = 6  # @param {type:"slider", min:3, max:10, step:1}
VIDEO_STYLE = "Informative & Educational"  # @param ["Informative & Educational", "Entertaining & Fun", "Dramatic & Cinematic", "Minimalist & Clean"]
# @markdown ---
# @markdown ### 🎤 Narrator Voice
VOICE = "en-US-AriaNeural"  # @param ["en-US-AriaNeural", "en-US-GuyNeural", "en-US-JennyNeural", "en-GB-SoniaNeural", "en-AU-NatashaNeural"]
# @markdown ---
# @markdown ### 🎨 Image Style Keywords
IMAGE_STYLE = "photorealistic, high quality, cinematic lighting, 4k"  # @param {type:"string"}

if not GEMINI_API_KEY.strip():
    print("⚠️ Enter your Gemini API key above, then re-run this cell.")
else:
    print("✅ Config saved!")
    print(f"   Topic: {VIDEO_TOPIC}")
    print(f"   Scenes: {NUM_SCENES} | Style: {VIDEO_STYLE}")
    print(f"   Voice: {VOICE}")

In [ ]:
# @title 🔧 Initialize Libraries
import os, asyncio, json, textwrap, warnings
warnings.filterwarnings("ignore")

import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import torch
import nest_asyncio
nest_asyncio.apply()

import google.generativeai as genai
import edge_tts

OUTPUT_DIR = Path("/content/rufus_output")
OUTPUT_DIR.mkdir(exist_ok=True)

genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel("gemini-1.5-flash")

GPU_AVAILABLE = torch.cuda.is_available()
if GPU_AVAILABLE:
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {name} ({vram:.1f} GB) — AI images enabled")
else:
    print("⚠️ No GPU — gradient placeholders will be used")
    print("   For AI images: Runtime → Change runtime type → T4 GPU")

print(f"✅ Output folder: {OUTPUT_DIR}")

In [ ]:
# @title 📝 Step 1: Generate Script

STYLE_MAP = {
    "Informative & Educational": "educational, clear, authoritative",
    "Entertaining & Fun": "entertaining, upbeat, humorous",
    "Dramatic & Cinematic": "dramatic, emotional, cinematic",
    "Minimalist & Clean": "minimalist, clean, concise"
}
style_desc = STYLE_MAP.get(VIDEO_STYLE, "informative")
print(f"Generating: {VIDEO_TOPIC!r} | {NUM_SCENES} scenes | {VIDEO_STYLE}")

prompt = (
    "You are a YouTube video scriptwriter.\n\n"
    f"Write a {NUM_SCENES}-scene script for: {VIDEO_TOPIC!r}\n"
    f"Tone: {style_desc}\nImage style: {IMAGE_STYLE}\n\n"
    "Return ONLY valid JSON (no markdown code blocks):\n"
    "{\n"
    "  \"title\": \"YouTube title under 70 chars\",\n"
    "  \"description\": \"150-200 word SEO description\",\n"
    "  \"tags\": [\"tag1\", \"tag2\", \"...(10 tags)\"],\n"
    "  \"scenes\": [\n"
    "    {\n"
    "      \"scene_number\": 1,\n"
    "      \"narration\": \"Spoken text 20-35 words\",\n"
    "      \"image_prompt\": \"Detailed visual description\",\n"
    "      \"subtitle\": \"Short subtitle max 8 words\"\n"
    "    }\n"
    "  ]\n"
    "}\n\n"
    "Rules:\n"
    "- Scene 1: Strong hook\n"
    f"- Scene {NUM_SCENES}: Call-to-action (like & subscribe)\n"
    "- Narrations: natural speech 20-35 words each\n"
    "- Image prompts: specific visuals, no text in images\n"
)

response = gemini.generate_content(prompt)
raw = response.text.strip()
if raw.startswith("```"):
    raw = "\n".join(raw.split("\n")[1:])
    if raw.startswith("json"):
        raw = raw[4:]
    if raw.endswith("```"):
        raw = raw[:-3]
raw = raw.strip()

try:
    script = json.loads(raw)
    print(f"\n✅ Script ready!\n")
    print(f"📺 {script['title']}")
    for s in script["scenes"]:
        n, sub, narr = s["scene_number"], s["subtitle"], s["narration"]
        preview = narr[:85] + "..." if len(narr) > 85 else narr
        print(f"\n  [{n}] {sub}")
        print(f"       \"{preview}\"")
except json.JSONDecodeError as e:
    print(f"❌ JSON error: {e}\n{raw[:400]}")
    raise ValueError("Re-run this cell to try again.")

In [ ]:
# @title 🔊 Step 2: Generate Voiceover

async def tts(text, voice, path):
    comm = edge_tts.Communicate(text, voice)
    await comm.save(str(path))

print(f"Voice: {VOICE}\n" + "-" * 45)
for i, s in enumerate(script["scenes"]):
    p = OUTPUT_DIR / f"audio_{i:02d}.mp3"
    asyncio.get_event_loop().run_until_complete(tts(s["narration"], VOICE, p))
    s["audio_path"] = str(p)
    wc = len(s["narration"].split())
    print(f"✅ Scene {i+1}/{len(script['scenes'])}: {wc} words → {p.name}")

print(f"\n✅ Voiceover complete!")

In [ ]:
# @title 🎨 Step 3: Generate Scene Images
import random

def make_placeholder(w=1280, h=720):
    palettes = [
        [(15,15,50),(80,20,120)], [(20,50,20),(40,100,60)],
        [(60,15,15),(120,50,20)], [(15,35,80),(20,70,130)],
    ]
    c1, c2 = random.choice(palettes)
    img = Image.new("RGB", (w, h))
    px = []
    for y in range(h):
        for x in range(w):
            s = x / w
            px.append(tuple(min(255, int(c1[k]*(1-s) + c2[k]*s)) for k in range(3)))
    img.putdata(px)
    return img

if GPU_AVAILABLE:
    from diffusers import StableDiffusionPipeline
    print("Loading SD v1.5 (~4 GB, one-time download)...")
    pipe = StableDiffusionPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        torch_dtype=torch.float16,
        safety_checker=None,
        requires_safety_checker=False
    )
    pipe = pipe.to("cuda")
    pipe.enable_attention_slicing()
    print("✅ Model loaded\n")
else:
    print("ℹ️ No GPU — generating gradient placeholders\n")

for i, s in enumerate(script["scenes"]):
    p = OUTPUT_DIR / f"image_{i:02d}.jpg"
    if GPU_AVAILABLE:
        with torch.no_grad():
            img = pipe(
                s["image_prompt"],
                negative_prompt="blurry, low quality, text, watermark, distorted",
                num_inference_steps=25,
                guidance_scale=7.5,
                width=896, height=512
            ).images[0].resize((1280, 720), Image.LANCZOS)
        tag = "AI"
    else:
        img = make_placeholder()
        tag = "Placeholder"
    img.save(str(p), "JPEG", quality=95)
    s["image_path"] = str(p)
    print(f"✅ Scene {i+1}/{len(script['scenes'])} [{tag}]: {p.name}")

print(f"\n✅ All images ready!")

In [ ]:
# @title 🎞️ Step 4: Assemble Video
from moviepy.editor import ImageClip, AudioFileClip, concatenate_videoclips

def add_subtitle(img_path, text, out_path, W=1280, H=720):
    img = Image.open(str(img_path)).convert("RGBA").resize((W, H), Image.LANCZOS)
    ov = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    draw = ImageDraw.Draw(ov)
    fs = 34
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", fs)
    except Exception:
        font = ImageFont.load_default()
    lines = textwrap.wrap(text, width=56)
    lh = fs + 8
    bh = len(lines) * lh + 28
    draw.rectangle([(0, H - bh), (W, H)], fill=(0, 0, 0, 175))
    y = H - bh + 14
    for line in lines:
        try:
            bbox = draw.textbbox((0, 0), line, font=font)
            tw = bbox[2] - bbox[0]
        except AttributeError:
            tw = draw.textsize(line, font=font)[0]
        x = (W - tw) / 2
        draw.text((x+2, y+2), line, font=font, fill=(0, 0, 0, 210))
        draw.text((x, y),   line, font=font, fill=(255, 255, 255, 255))
        y += lh
    Image.alpha_composite(img, ov).convert("RGB").save(str(out_path), "JPEG", quality=92)

print("Building clips...")
clips = []
total_dur = 0.0
for i, s in enumerate(script["scenes"]):
    fp = OUTPUT_DIR / f"frame_{i:02d}.jpg"
    add_subtitle(s["image_path"], s["narration"], fp)
    audio = AudioFileClip(s["audio_path"])
    clip = ImageClip(str(fp)).set_duration(audio.duration).set_audio(audio)
    clips.append(clip)
    total_dur += audio.duration
    sub = s["subtitle"]
    print(f"✅ Scene {i+1}: {audio.duration:.1f}s — {sub}")

OUTPUT_VIDEO = "/content/rufus_video.mp4"
print(f"\nRendering {total_dur:.1f}s video...")
final = concatenate_videoclips(clips, method="compose")
final.write_videofile(
    OUTPUT_VIDEO, fps=24, codec="libx264", audio_codec="aac",
    temp_audiofile="/content/tmp_audio.m4a", remove_temp=True, logger=None
)
print(f"\n🎉 Done! {total_dur:.1f}s | {OUTPUT_VIDEO}")

In [ ]:
# @title 🎉 Step 5: Preview & Download
from IPython.display import Video, display
from google.colab import files
import os

size_mb = os.path.getsize(OUTPUT_VIDEO) / 1e6
title = script["title"]
desc = script["description"]
tags = ", ".join(script["tags"][:6])
n_scenes = len(script["scenes"])

print("=" * 50)
print("🎬 VIDEO READY!")
print("=" * 50)
print(f"\n📺 {title}")
print(f"⏱️  {total_dur:.1f}s | 📁 {size_mb:.1f} MB | 🎬 {n_scenes} scenes")
print(f"\n🏷️  Tags: {tags}")
print(f"\n📝 Description:\n{desc[:300]}...")
print("\n📥 Downloading...")
files.download(OUTPUT_VIDEO)
print("\n▶️  Preview:")
display(Video(OUTPUT_VIDEO, embed=True, width=800, height=450))